# MP06 — Low-Resolution Image Generator

Expanded engineering lesson; learner mastery unassessed.

## Intuition, definitions and worked mathematics
A spatial scale s=2 or 4 means LR height H/s and width W/s, not H/s². Pixel count decreases by s². C bands remain unchanged. For synthetic pairing, HR and LR originate from the same scene and declared crop; a geographic split must happen before extracting patches.

The model y=D_s(k*x) means first blur x with kernel k, then sample with operator D_s. Unfiltered phase-zero decimation takes x[0::s,0::s]. Area averaging instead takes the mean of each s×s block. They are different forward models. Blur before decimation suppresses high spatial frequencies that would fold into lower frequencies. A filter is not a guarantee that all aliasing is removed.

For [[0,2],[4,6]], factor-two area averaging gives 3; phase-zero decimation gives 0. For a one-pixel checkerboard, decimation at an even factor produces a constant depending on sampling phase, although the original oscillates. This is aliasing. The mean of each 2×2 cell is 0.5, which describes local coverage but cannot recover the missing fine pattern.

A valid pair manifest needs source/checksum, crop, split, band order, scale, operator and file hashes. Changing any of these changes the experimental meaning. Our generator rejects dimensions not divisible by s instead of silently cropping.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
root=Path.cwd()
if not (root/'src').exists(): root=root.parent
sys.path.insert(0,str(root/'src'))
from rssr.degradation_lab import gaussian,filter_hwc,degrade_hwc,spectrum
x=np.array([[0.,2.],[4.,6.]])[...,None]
assert degrade_hwc(x,2,0,0)[0,0,0]==3
assert x[::2,::2][0,0,0]==0
checker=(np.indices((16,16)).sum(0)%2)[...,None]
for scale in (2,4):
    print(scale,checker[::scale,::scale].mean(),degrade_hwc(checker,scale,0).mean(),degrade_hwc(checker,scale,.8).shape)

## Real-data experiment and interpretation
The measured training-crop manifest contains explicit unfiltered decimation pairs and separate area-average pairs, with optional Gaussian blur, at ×2 and ×4. All pairs keep four bands. Shape and checksum records make the transformation reproducible. This is synthetic degradation of an aerial image; it does not create a real paired satellite dataset.

Run `python scripts/degradation_experiments.py` explicitly from the repository root. The notebook below reads recorded measurements; it does not download data or execute a hidden research run. The source is a checksummed 64×64 NAIP training crop with fixed stored-value/10000 scaling, not verified reflectance.

In [ ]:
p=root/'reports/mp05-mp08-experiments.json'
if p.exists():
    record=json.loads(p.read_text())
    print(record['mp06_mp07_pairs'])
else: print('Real experiment has not run in this checkout.')

## Practice — attempt before reading solutions
1. What is the LR shape for HR 64×64×4 at ×4?
2. Why split scenes before patches?
3. Does sigma=0 in degrade_hwc mean no antialias filtering?

## Separate solution guide
1. 16×16×4; 256 spatial pixels rather than 4096.
2. Adjacent patches and repeated locations could otherwise leak between training and evaluation.
3. No. This function still area-averages. The separate decimation pairs are the unfiltered control.

These are generated examples, not evidence that a learner solved the exercises.

In [ ]:
assert degrade_hwc(np.ones((64,64,4)),4).shape==(16,16,4)

## Reuse, execution and limitations
Use `src/rssr/degradation_lab.py` and `python -m pytest tests/test_degradation_lab.py`. Follow [local and Colab setup](../docs/setup.md); run from notebooks/ or the repository root. The shared NumPy code preserves HWC channels and uses explicit boundaries. No physical sensor calibration or geographic generalization is established.

References: the authoritative stage brief, [dataset card](../docs/dataset-cards.md), [protocol](../docs/protocol.md), and [NumPy FFT documentation](https://numpy.org/doc/stable/reference/routines.fft.html). See [execution evidence](../reports/notebook-execution.json) and [experiment record](../reports/mp05-mp08-experiments.json).